# Data

In [37]:
import nltk
from nltk.corpus import gutenberg, stopwords
from nltk import tokenize
import autocorrect
from nltk.stem import WordNetLemmatizer
from autocorrect import Speller
nltk.download('omw-1.4')
import regex
from gensim.models import Word2Vec

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\kashk\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [24]:
# Assigning the plays to variables
hamlet = gutenberg.raw('shakespeare-hamlet.txt')
macbeth = gutenberg.raw('shakespeare-macbeth.txt')
julius_caesar = gutenberg.raw('shakespeare-caesar.txt')

In [25]:
# Keeping all the text in a single variable with normalized case
lower_text = (hamlet + macbeth + julius_caesar).lower()

In [26]:
# Tokenizing sentences
sentences = tokenize.sent_tokenize(lower_text)

In [27]:
# Tokenizing words
words = [tokenize.word_tokenize(sent) for sent in sentences]

In [28]:
# Correcting spelling errors
spell = Speller(lang = 'en')
correct_sentences = [[spell(word) for word in sentence] for sentence in words]

In [29]:
# Removing stop words
stop_words = set(stopwords.words('english'))
no_stop_words = [[word for word in sentence if word not in stop_words] for sentence in correct_sentences]

In [30]:
# Lemmatizing
lemmatizer = WordNetLemmatizer()
lemmatized_words = [[lemmatizer.lemmatize(word) for word in sentence] for sentence in no_stop_words]

In [33]:
# Removing punctuation and then the empty tokens created by the cleanup
regex_cleaned = [[regex.sub(r'[^a-zA-Z0-9]', '', word) for word in sentence] for sentence in lemmatized_words]
regex_cleaned = [[word for word in sentence if word.strip() != ''] for sentence in regex_cleaned]

In [34]:
# Printing the first five sentences
for i in range(5):
    print(f"Sentence {i}: {regex_cleaned[i]}")

Sentence 0: ['tragedy', 'hamlet', 'william', 'shakespeare', '1599', 'act', 'prime']
Sentence 1: ['scene', 'prima']
Sentence 2: ['enter', 'bernard', 'francisco', 'two', 'sentinel']
Sentence 3: ['bernard']
Sentence 4: ['s']


In [35]:
# Comparing with the actual few sentences
for i in range(5):
    print(f"Sentence {i}: {words[i]}")

Sentence 0: ['[', 'the', 'tragedie', 'of', 'hamlet', 'by', 'william', 'shakespeare', '1599', ']', 'actus', 'primus', '.']
Sentence 1: ['scoena', 'prima', '.']
Sentence 2: ['enter', 'barnardo', 'and', 'francisco', 'two', 'centinels', '.']
Sentence 3: ['barnardo', '.']
Sentence 4: ['who', "'s", 'there', '?']


# Processing

## Word2Vec models

### CBOW: target word from context words

In [38]:
model = Word2Vec(sentences=regex_cleaned, vector_size=100, window=5, min_count=3, workers=4, sg=0, epochs=20)

In [39]:
top20 = list(model.wv.key_to_index.keys())[:20]

print("Top 20 most frequent words and their counts:\n")
for word in top20:
    count = model.wv.get_vecattr(word, "count")
    print(f"{word:15}  {count}")

Top 20 most frequent words and their counts:

d                585
ham              337
thou             307
lord             306
shall            300
s                295
come             284
king             248
enter            230
good             221
let              220
mac              205
thy              202
like             200
cesar            193
one              188
make             185
know             184
v                175
thee             174


Based on this word count, we notice a few things:
- d is the top word, which possibly came from words I'd, you'd, we'd. It is just a contraction of 'would', and should probably be removed. Similarly, s and v perhaps correspond to 'is' and 'have'.
- Archaic pronouns like 'thee', 'thy', 'thou' constitute stop words and should also be removed.
- 'Caesar', 'Hamlet' and 'Macbeth' being correct to 'cesar', 'mac', 'ham' sounds lowkey appetizing and defeats the purpose. However, we cannot remove the spelling correction step because the text consists of Latin words which should be converted to English. A better decision might be to manually code those words to preserve the original meaning.

In [43]:
archaic = {"thou", "thee", "thy", "thine", "ye", "hath", "doth"}
stop_words.update(archaic)

In [44]:
regex_cleaned = [[w for w in sent if len(w) > 1] for sent in regex_cleaned]

In [45]:
name_map = {
    "hamlet": "hamlet",
    "ham": "hamlet",
    "macbeth": "macbeth",
    "mac": "macbeth",
    "caesar": "caesar",
    "cesar": "caesar"
}

normalized = [[name_map.get(w, w) for w in sent] for sent in regex_cleaned]